# Phase 4: Confidence-Calibrated Selective Prediction & Structured Conflict Cards (v2)
### IPC2BNS-Verify: Zero-Risk Selective Prediction & Proactive User Clarification

This notebook demonstrates:
1. **Confidence Threshold Gating ($\tau = 0.80$)**: Refuses to output false-certainty hallucinations when certainty is $< 80\%$.
2. **Structured Conflict Disclosure Cards**: Emits actionable multi-jurisdiction summaries when High Courts are actively split.
3. **Proactive Clarification Prompts**: Requests missing incident/FIR dates when procedural law is ambiguous.

In [ ]:
# ==============================================================================
# STEP 0: GOOGLE COLAB & GOOGLE DRIVE INITIALIZATION (RUN THIS FIRST)
# ==============================================================================
import os, sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
    print("[Colab] Detected Google Colab environment. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    
    # Check all possible Drive directory variations
    drive_candidates = [
        Path('/content/drive/MyDrive/NLP_rspaper'),
        Path('/content/drive/MyDrive/NLP-rspaper'),
        Path('/content/drive/MyDrive/research paper/NLP_rs'),
        Path('/content/drive/MyDrive/NLP_rs'),
        Path.cwd()
    ]
    
    found_dir = None
    for cand in drive_candidates:
        if (cand / 'code' / 'src').exists():
            found_dir = cand
            break
            
    if not found_dir:
        # Search inside MyDrive
        print("[Colab] Searching Google Drive for project folder...")
        for match in Path('/content/drive/MyDrive').glob('**/code/src'):
            found_dir = match.parent.parent
            break
            
    if found_dir:
        os.chdir(found_dir)
        print(f"[Colab] Working directory set to: {os.getcwd()}")
        sys.path.insert(0, str(found_dir / 'code'))
    else:
        print("[Colab] Warning: Could not locate project directory in Google Drive. Using:", os.getcwd())
        
    if Path("requirements.txt").exists():
        print("[Colab] Installing dependencies from requirements.txt...")
        get_ipython().system("pip install -q -r requirements.txt")
except ImportError:
    IN_COLAB = False
    print("[Local] Running in local environment:", os.getcwd())
    root_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'code']
    for p in root_candidates:
        if (p / 'code' / 'src').exists():
            sys.path.insert(0, str(p / 'code'))
            break
        elif (p / 'src').exists():
            sys.path.insert(0, str(p))
            break

print("[Setup Complete] sys.path[0]:", sys.path[0])
print("[Setup Complete] Current working directory:", os.getcwd())


## 1. Import Selective Prediction Engine

In [ ]:
# --- Self-Healing Colab Environment Bootstrap ---
import os, sys
from pathlib import Path

# 1. Check & insert paths where 'src' is located
candidates = [
    Path.cwd() / 'code',
    Path.cwd(),
    Path('/content/drive/MyDrive/NLP_rspaper/code'),
    Path('/content/drive/MyDrive/NLP_rspaper'),
    Path('/content/drive/MyDrive/NLP-rspaper/code'),
    Path('/content/drive/MyDrive/NLP-rspaper'),
    Path('/content/drive/MyDrive/research paper/NLP_rs/code'),
    Path('/content/drive/MyDrive/research paper/NLP_rs')
]

for p in candidates:
    if (p / 'src').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        if os.getcwd() != str(p.parent if p.name == 'code' else p):
            try:
                os.chdir(str(p.parent if p.name == 'code' else p))
            except Exception:
                pass
        break
# ------------------------------------------------

from src.temporal.abstention_engine import SelectivePredictionEngine

abstention_eng = SelectivePredictionEngine(confidence_threshold=0.80)
print("SelectivePredictionEngine initialized with tau=0.80!")

## 2. Test Conflict Card Generation & Missing Date Clarification

In [ ]:
# Example 1: Missing Dates on Procedurally Sensitive Posture
q_missing = "Filing bail application for an accused charged with cheating."
card1 = abstention_eng.evaluate_query(q_missing)
print("=== TEST 1: Missing Dates ===")
print("Should Abstain:", card1.should_abstain)
print("Reason:", card1.abstention_reason)
print("Clarification Questions:", card1.required_clarifications)

# Example 2: Unresolved High Court Split without Specified Jurisdiction
q_split = "Trial convicted appellant under IPC in May 2024. Filing criminal appeal in August 2024."
card2 = abstention_eng.evaluate_query(q_split)
print("\n=== TEST 2: High Court Split Card ===")
print("Should Abstain:", card2.should_abstain)
print("Conflict Type:", card2.conflict_type)
print("Recommendation:", card2.safe_recommendation)

# Example 3: Confident Unambiguous Query
q_clear = "Offence of theft committed on 10 August 2024 with FIR on 12 August 2024."
card3 = abstention_eng.evaluate_query(q_clear)
print("\n=== TEST 3: Confident Prediction ===")
print("Should Abstain:", card3.should_abstain)
print("Confidence Score:", card3.confidence_score)

## 3. Verify Pytest Abstention Suite

In [ ]:
!pytest code/tests/test_abstention.py -v